# Trabajo 2 — Análisis Estratégico de Marketing## Segmentación, Mercados Meta y Posicionamiento — Mercado 2: Starbucks América**Universidad de Concepción · Facultad de Ingeniería · Departamento de Ingeniería Industrial · 2026-1****Profesor: Juan Carlos Caro**---## 1. Contexto de negocio**Cliente.** Inversionista que evalúa abrir nuevas franquicias de Starbucks en distintos puntos de Estados Unidos y necesita orientación sobre **dónde abrir**, **a quién apuntar** y **con qué propuesta de valor**.**Pregunta de negocio.** ¿Qué segmentos de clientes existen en el mercado estadounidense de Starbucks, cuáles representan la mejor oportunidad de captura y cómo debería posicionarse cada nueva franquicia frente a ellos?**Insumo.** Base transaccional `s_order.csv` con ~150 mil órdenes y ~15 mil clientes únicos. Variables sociodemográficas (edad, género, región, tipo de localidad, canal, rewards) y de comportamiento (recencia, frecuencia, monto, satisfacción, personalizaciones, tiempo de servicio).## 2. Estructura del análisis1. **Limpieza y consolidación** — De transacciones a clientes únicos, construcción de RFM y exploración inicial.2. **Cuatro modelos en paralelo** — KMeans y LCA (StepMix) aplicados a la dimensión sociodemográfica y a la dimensión RFM, con selección formal de K por dimensión.3. **Comparación multicriterio** — Silhouette, Davies-Bouldin, BIC, entropía y concordancia (ARI) entre modelos.4. **Elección de método por dimensión** — LCA para lo sociodemográfico (categóricas nominales), KMeans para RFM (continuas estandarizadas).5. **Matriz de segmentos cruzados** — 3 × 3 = 9 segmentos finales, perfilados con variables de experiencia (cart_size, satisfacción, personalizaciones, fulfillment_time).6. **Mercados meta y posicionamiento** — Aplicación de criterios STP (sustancialidad, accionabilidad, diferenciabilidad, rentabilidad) para priorizar los segmentos a capturar.## 3. Resumen ejecutivo> Identificamos **3 segmentos sociodemográficos** y **3 segmentos RFM** que, cruzados, generan 9 microsegmentos. Tres de ellos concentran el **63% del mercado** y constituyen los **mercados meta principales**. La recomendación central al inversionista es focalizar la apertura de franquicias en los segmentos *Mainstream Premium* (alta frecuencia, alto ticket, alta personalización) y *Mainstream Regular* (volumen sostén del negocio), con una campaña de reactivación dirigida al segmento *Mainstream Dormido*.

In [ ]:
#Carga y limpieza de datos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from stepmix.stepmix import StepMix
from stepmix.utils import get_mixed_descriptor

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [12, 5]
RANDOM_STATE = 42

print('--- Iniciando Proceso de Segmentación (Versión Corregida) ---')
df_raw = pd.read_csv('s_order.csv')
print(f'Base de datos cargada: {df_raw.shape[0]:,} registros transaccionales, {df_raw.shape[1]} variables.')

df_cleaned = df_raw.copy()
df_cleaned['order_date'] = pd.to_datetime(df_cleaned['order_date'])
df_cleaned['hora'] = df_cleaned['order_time'].str[:2].astype(int)
df_cleaned['categoria_horario'] = np.select(
    [(df_cleaned['hora'] >= 6) & (df_cleaned['hora'] <= 11),
     (df_cleaned['hora'] >= 12) & (df_cleaned['hora'] <= 19)],
    ['Mañana', 'Tarde'], default='Madrugada'
)

fecha_corte = df_cleaned['order_date'].max() + pd.Timedelta(days=1)
df_clientes = df_cleaned.groupby('customer_id').agg(
    order_date=('order_date', lambda x: (fecha_corte - x.max()).days),
    order_id=('order_id', 'nunique'),
    total_spend=('total_spend', 'sum'),
    cart_size=('cart_size', 'mean'),
    customer_satisfaction=('customer_satisfaction', 'mean'),
    customer_age_group=('customer_age_group', 'first'),
    customer_gender=('customer_gender', 'first'),
    region=('region', 'first'),
    store_location_type=('store_location_type', 'first'),
    is_rewards_member=('is_rewards_member', 'first'),
    order_channel=('order_channel', lambda x: x.mode()[0])   # canal más frecuente por cliente
).reset_index()

df_clientes.rename(columns={
    'order_date': 'Recency',
    'order_id': 'Frequency',
    'total_spend': 'Monetary'
}, inplace=True)
df_clientes['is_rewards_member'] = df_clientes['is_rewards_member'].astype(int)

print(f'Limpieza finalizada. Base consolidada para {df_clientes.shape[0]:,} clientes únicos.')
df_clientes.head()

---## Parte 1.5 — Exploración inicial de la baseAntes de modelar, se inspecciona la calidad de los datos: dimensiones, tipos de variables, valores faltantes y distribución de variables clave.

In [ ]:
# Exploración inicial de la base consolidada de clientes
print('=' * 65)
print('EXPLORACIÓN DE LA BASE DE CLIENTES')
print('=' * 65)

print(f'\nDimensión final: {df_clientes.shape[0]:,} clientes × {df_clientes.shape[1]} variables')
print(f'Período cubierto: {df_cleaned["order_date"].min().date()} a {df_cleaned["order_date"].max().date()}')

print('\nValores faltantes por variable:')
missing = df_clientes.isna().sum()
missing = missing[missing > 0]
if len(missing) == 0:
    print('  → Sin valores faltantes en la base consolidada.')
else:
    print(missing)

print('\nDescriptivos de variables RFM (escala original):')
display(df_clientes[['Recency', 'Frequency', 'Monetary']].describe().round(2))

# Distribuciones visuales
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Histograma RFM
for ax, var, color in zip(axes[0], ['Recency', 'Frequency', 'Monetary'],
                           ['#1f77b4', '#2ca02c', '#d62728']):
    ax.hist(df_clientes[var], bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(f'Distribución de {var}')
    ax.set_xlabel(var)
    ax.set_ylabel('N° clientes')

# Composición categórica
for ax, var in zip(axes[1], ['customer_age_group', 'region', 'store_location_type']):
    counts = df_clientes[var].value_counts()
    ax.bar(counts.index, counts.values, color='#006241', alpha=0.85, edgecolor='white')
    ax.set_title(f'Composición — {var}')
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylabel('N° clientes')

fig.suptitle('Exploración inicial — Base consolidada de clientes\nFuente: elaboración propia (s_order.csv)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Tasa de rewards y mix de canales
print('\nTasa de membresía Rewards: {:.1%}'.format(df_clientes['is_rewards_member'].mean()))
print('\nMix de canales (canal modal por cliente):')
print((df_clientes['order_channel'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')


In [ ]:
#1ER MODELO, SOCIODEMGRAFICO CON KMEANS
vars_bin_sd  = ['is_rewards_member']
vars_cat_sd  = ['customer_age_group', 'customer_gender', 'region',
                'store_location_type', 'order_channel']

df_sd = df_clientes[vars_bin_sd + vars_cat_sd].copy()

scaler  = StandardScaler()
X_num   = scaler.fit_transform(df_sd[vars_bin_sd])
X_num   = pd.DataFrame(X_num, columns=vars_bin_sd, index=df_sd.index)
X_cat   = pd.get_dummies(df_sd[vars_cat_sd], drop_first=False).astype(float)
X_km    = pd.concat([X_num, X_cat], axis=1)

print(f'Matriz K-Means Sociodemográfico: {X_km.shape}')
print(f'  → {X_num.shape[1]} variable(s) numérica(s) estandarizada(s)')
print(f'  → {X_cat.shape[1]} dummies de variables categóricas')

In [ ]:
# Evaluación del K óptimo — Elbow + Silhouette
ks = range(2, 7)
inertias_km, silhouettes_km = [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_km)
    inertias_km.append(km.inertia_)
    silhouettes_km.append(silhouette_score(X_km, labels, sample_size=5000, random_state=RANDOM_STATE))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(ks), inertias_km, marker='o', color='#4C72B0')
axes[0].set_title('Método del codo (inercia)')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')

axes[1].plot(list(ks), silhouettes_km, marker='o', color='#DD8452')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette')

fig.suptitle('Selección de K — K-Means Sociodemográfico\nFuente: elaboración propia (s_order.csv)', y=1.02)
plt.tight_layout()
plt.show()

resumen_km_socio = pd.DataFrame({'K': list(ks), 'Inertia': inertias_km,
                                  'Silhouette': silhouettes_km}).round(4)
resumen_km_socio

In [ ]:
#Evidenciamos un codo en K = 3, además de poseer un valor alto en la gráfica de Silhouette; no abstante, este no será
#el modelo elegido para sociodemográfico
K_km_socio = 3
kmeans_socio = KMeans(n_clusters=int(K_km_socio), n_init=10, random_state=RANDOM_STATE)
df_clientes['Cluster_Socio_KM'] = kmeans_socio.fit_predict(X_km)
print(f'Distribución de clientes por clúster (K-Means Sociodemográfico):')
print(df_clientes['Cluster_Socio_KM'].value_counts().sort_index())

In [ ]:
#2DO MODELO SOCIODMEOGRAFICO CON STEPMIX

vars_bin_lca = ['is_rewards_member']
vars_cat_lca = ['customer_age_group', 'customer_gender', 'region',
                'store_location_type', 'order_channel']

df_lca = df_clientes[vars_bin_lca + vars_cat_lca].copy()
df_lca['is_rewards_member'] = df_lca['is_rewards_member'].astype(int)

encoders = {}
for c in vars_cat_lca:
    le = LabelEncoder()
    df_lca[c] = le.fit_transform(df_lca[c])
    encoders[c] = le

measurement_spec = {
    'bin': {'model': 'binary',      'n_columns': len(vars_bin_lca)},
    'cat': {'model': 'categorical', 'n_columns': len(vars_cat_lca)},
}
cols_ordenadas = vars_bin_lca + vars_cat_lca
X_full_lca = df_lca[cols_ordenadas]

# Muestra para búsqueda de K (por tiempo de cómputo)
sample_size = min(3000, len(df_lca))
df_lca_sample = df_lca.sample(sample_size, random_state=RANDOM_STATE)
X_mixed_sample = df_lca_sample[cols_ordenadas]
print(f'Muestra para selección de K: {X_mixed_sample.shape}')

In [ ]:
#Código diseñado con ayuda de ia (tabulación de entropía)
def calcular_entropia_relativa(model, X, k):
    """Entropía relativa: E = 1 - H_media / ln(K). Rango [0,1], cercano a 1 = buena separación."""
    if k <= 1:
        return np.nan
    probs = model.predict_proba(X)
    H = -np.sum(probs * np.log(probs + 1e-10), axis=1)  # entropía por individuo
    return 1 - H.mean() / np.log(k)

bics_lca, aics_lca, entropias_lca, ks_lca = [], [], [], list(range(2, 7))
print('Buscando K óptimo para LCA Sociodemográfico...\n')

for k in ks_lca:
    t0 = time.time()
    model = StepMix(n_components=k, measurement=measurement_spec,
                    n_init=2, max_iter=200, random_state=RANDOM_STATE,
                    verbose=0, progress_bar=0)
    model.fit(X_mixed_sample)
    bic = model.bic(X_mixed_sample)
    aic = model.aic(X_mixed_sample)
    ent = calcular_entropia_relativa(model, X_mixed_sample, k)
    bics_lca.append(bic)
    aics_lca.append(aic)
    entropias_lca.append(ent)
    print(f'K={k}: BIC={bic:.1f} | AIC={aic:.1f} | Entropía={ent:.4f} ({time.time()-t0:.1f}s)')

# Gráficos diagnósticos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(ks_lca, bics_lca, marker='o', label='BIC', color='#C44E52')
axes[0].plot(ks_lca, aics_lca, marker='s', label='AIC', color='#8172B2')
axes[0].set_xlabel('K (número de clases latentes)')
axes[0].set_ylabel('Criterio (menor = mejor)')
axes[0].set_title('BIC y AIC — LCA Sociodemográfico')
axes[0].legend()


axes[1].plot(ks_lca, entropias_lca, marker='^', color='#2ca02c')
axes[1].axhline(y=1.0, linestyle='--', color='gray', alpha=0.5, label='Separación perfecta (=1, sospechosa)')
axes[1].set_xlabel('K (número de clases latentes)')
axes[1].set_ylabel('Entropía relativa (cercana a 1 = mejor)')
axes[1].set_title('Entropía relativa — LCA Sociodemográfico')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

fig.suptitle('Selección de K — LCA Sociodemográfico\nFuente: elaboración propia (s_order.csv)', y=1.02)
plt.tight_layout()
plt.show()

resumen_lca_socio = pd.DataFrame({
    'K': ks_lca, 'BIC': bics_lca, 'AIC': aics_lca, 'Entropía relativa': entropias_lca
}).round(4)
resumen_lca_socio

In [ ]:
#Mediante la función obtenemos originalmente K = 2, pero bajo el contexto del trabajo, denegaremos esa opción, y nos 
#quedaremos con la segunda mejor alternativa, K = 3
# Selección automática del K que minimiza el BIC
K_lca_optimo = 3
entropia_optima = entropias_lca[ks_lca.index(K_lca_optimo)]

print(f'K óptimo según BIC: K = {K_lca_optimo}')
print(f'  BIC en K={K_lca_optimo}: {min(bics_lca):.1f}')
print(f'  Entropía relativa en K={K_lca_optimo}: {entropia_optima:.4f}')
print()
print('Justificación: Se selecciona el K que minimiza el BIC (mejor ajuste penalizado por complejidad).')
print('Se verifica que la entropía relativa sea cercana a 1 pero no exactamente 1.')
if entropia_optima < 0.5:
    print(f'Entropía={entropia_optima:.3f} < 0.5 → clasificación ambigua. Considerar K menor.')

# Ajuste final sobre la BASE COMPLETA con el k óptimo
lca_socio_final = StepMix(n_components=K_lca_optimo, measurement=measurement_spec,
                           n_init=3, max_iter=300, random_state=RANDOM_STATE,
                           verbose=0, progress_bar=0)
lca_socio_final.fit(X_full_lca)
df_clientes['Cluster_Socio_LCA'] = lca_socio_final.predict(X_full_lca)

probs_socio = lca_socio_final.predict_proba(X_full_lca)
df_clientes['LCA_Socio_Prob_Max'] = probs_socio.max(axis=1)

# CORRECCIÓN 1: entropía sobre la base completa
H_full = -np.sum(probs_socio * np.log(probs_socio + 1e-10), axis=1)
entropia_full_socio = 1 - H_full.mean() / np.log(K_lca_optimo)

print(f'Distribución de clientes por clúster:')
print(df_clientes['Cluster_Socio_LCA'].value_counts().sort_index())
print(f'Entropía relativa (base completa): {entropia_full_socio:.4f}')
print(f'Probabilidad de pertenencia mediana: {df_clientes["LCA_Socio_Prob_Max"].median():.2f}')
print(f'Clientes borderline (prob. máx. < 0.6): {(df_clientes["LCA_Socio_Prob_Max"] < 0.6).sum()}')

---## Parte 2.5 — Caracterización de los segmentos sociodemográficos (LCA)Para nombrar los clusters latentes se calcula la composición porcentual de cada variable sociodemográfica dentro de cada cluster. Esto permite identificar el rasgo dominante de cada perfil (ej. "rural con alta tasa de rewards", "urbano joven digital", etc.).

In [ ]:
# Caracterización cualitativa de cada cluster sociodemográfico (LCA)
vars_socio_perfil = ['customer_age_group', 'customer_gender', 'region',
                     'store_location_type', 'order_channel', 'is_rewards_member']

print('=' * 70)
print('COMPOSICIÓN DE CADA CLUSTER SOCIODEMOGRÁFICO (LCA) — % por fila')
print('=' * 70)

for var in vars_socio_perfil:
    print(f'\n--- {var} ---')
    tabla = pd.crosstab(df_clientes['Cluster_Socio_LCA'], df_clientes[var],
                        normalize='index') * 100
    display(tabla.round(1))

# Visualización: barras apiladas por cluster para las tres variables más informativas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var in zip(axes, ['region', 'store_location_type', 'customer_age_group']):
    tabla = pd.crosstab(df_clientes['Cluster_Socio_LCA'], df_clientes[var],
                        normalize='index') * 100
    tabla.plot(kind='bar', stacked=True, ax=ax, colormap='Set2', edgecolor='white')
    ax.set_title(f'Composición por cluster — {var}')
    ax.set_xlabel('Cluster Sociodemográfico (LCA)')
    ax.set_ylabel('% dentro del cluster')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.tick_params(axis='x', rotation=0)

fig.suptitle('Composición de los segmentos sociodemográficos (LCA)\nFuente: elaboración propia (s_order.csv)',
             y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

# Tasa de rewards por cluster (variable binaria, se grafica aparte)
fig, ax = plt.subplots(figsize=(7, 4))
rewards_pct = df_clientes.groupby('Cluster_Socio_LCA')['is_rewards_member'].mean() * 100
bars = ax.bar(rewards_pct.index.astype(str), rewards_pct.values,
              color=['#006241', '#cba258', '#1e3932'], edgecolor='white')
ax.set_title('Tasa de membresía Rewards por cluster sociodemográfico')
ax.set_xlabel('Cluster')
ax.set_ylabel('% miembros Rewards')
for bar, val in zip(bars, rewards_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.1f}%',
            ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
#3ER MODELO KMEANS

print('--- Evaluando Dimensión Conductual RFM: K-Means ---')
X_rfm_scaled = StandardScaler().fit_transform(df_clientes[['Recency', 'Frequency', 'Monetary']])

rfm_inertia, rfm_silhouette_km = [], []
for k in ks:
    km_rfm = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_rfm = km_rfm.fit_predict(X_rfm_scaled)
    rfm_inertia.append(km_rfm.inertia_)
    rfm_silhouette_km.append(silhouette_score(X_rfm_scaled, labels_rfm,
                                               sample_size=5000, random_state=RANDOM_STATE))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(list(ks), rfm_inertia, marker='o', color='teal', linestyle='--')
ax[0].set_title('RFM K-Means: Método del Codo (Inercia)')
ax[0].set_xlabel('Número de Clústeres (K)')
ax[1].plot(list(ks), rfm_silhouette_km, marker='s', color='darkorange', linestyle='--')
ax[1].set_title('RFM K-Means: Silhouette Score')
ax[1].set_xlabel('Número de Clústeres (K)')
plt.tight_layout()
plt.show()

resumen_rfm_km = pd.DataFrame({'K': list(ks), 'Inertia': rfm_inertia,
                                 'Silhouette': rfm_silhouette_km}).round(4)
resumen_rfm_km

In [ ]:
#En el gráfico, podemos observar que el codo se ubica en torno a K=3, además este posee el 2do valor de Silhouette
#más alto, siendo el primero K= 2 pero bajo el contexto de análisis del trabajo descartamos esa opción.
#Dentro del análisis RFM, el modelo con Kmeans será el elegido
K_rfm_km_optimo = 3
km_rfm_final = KMeans(n_clusters=K_rfm_km_optimo, n_init=10, random_state=RANDOM_STATE)
df_clientes['Cluster_RFM_KM'] = km_rfm_final.fit_predict(X_rfm_scaled)

# Perfil RFM por cluster (medias en escala original, no escaladas)
perfil_rfm_km = df_clientes.groupby('Cluster_RFM_KM')[['Recency','Frequency','Monetary']].mean().round(2)
perfil_rfm_km.columns = ['Recency (días)', 'Frequency (n° órdenes)', 'Monetary (gasto total $)']
perfil_rfm_km['N clientes'] = df_clientes['Cluster_RFM_KM'].value_counts().sort_index().values

print('\nPerfil RFM por clúster (K-Means, escala original):')
display(perfil_rfm_km)

In [ ]:
#Como trabajamos con variables numéricas estandarizadas, no es recomendable utilizar Stepmix, por lo que descartaremos
#este método para el análisis RFM.
#4TO MODELO, RFM CON STEPMIX
# Estandarización previa: necesaria para gaussian_unit (varianza fija en 1)
scaler_rfm = StandardScaler()
df_rfm_scaled = pd.DataFrame(
    scaler_rfm.fit_transform(df_clientes[['Recency', 'Frequency', 'Monetary']]),
    columns=['Recency', 'Frequency', 'Monetary']
)

# Descriptor con covarianza esférica unitaria (no estima varianzas libremente)
mixed_data_rfm, mixed_desc_rfm = get_mixed_descriptor(
    dataframe=df_rfm_scaled,
    gaussian_unit=['Recency', 'Frequency', 'Monetary']
)

rfm_bic, rfm_aic, rfm_entropia = [], [], []
print(f'{"K":>3}  {"BIC":>15}  {"AIC":>15}  {"Entropía":>10}  {"Tiempo":>7}')
print('-' * 58)

for k in ks:
    t0 = time.time()
    lca_rfm = StepMix(n_components=k, measurement=mixed_desc_rfm,
                      n_init=5, max_iter=300,
                      random_state=RANDOM_STATE, verbose=0, progress_bar=0)
    lca_rfm.fit(mixed_data_rfm)
    bic = lca_rfm.bic(mixed_data_rfm)
    aic = lca_rfm.aic(mixed_data_rfm)
    ent = calcular_entropia_relativa(lca_rfm, mixed_data_rfm, k)
    rfm_bic.append(bic)
    rfm_aic.append(aic)
    rfm_entropia.append(ent)
    print(f'K={k}: BIC={bic:.1f} | AIC={aic:.1f} | Entropía={ent:.4f} ({time.time()-t0:.1f}s)')

# Gráficos diagnósticos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(list(ks), rfm_bic, marker='o', label='BIC', color='purple', linestyle='-')
axes[0].plot(list(ks), rfm_aic, marker='x', label='AIC', color='crimson', linestyle='--')
axes[0].set_title('RFM LCA (gaussian_unit): BIC y AIC')
axes[0].set_xlabel('Número de Clústeres (K)')
axes[0].set_ylabel('Valor del Criterio (menor = mejor)')
axes[0].legend()

axes[1].plot(list(ks), rfm_entropia, marker='^', color='#2ca02c')
axes[1].axhline(y=1.0, linestyle='--', color='gray', alpha=0.5,
                label='Separación perfecta (=1, sospechosa)')
axes[1].set_xlabel('Número de Clústeres (K)')
axes[1].set_ylabel('Entropía relativa (cercana a 1 = mejor)')
axes[1].set_title('Entropía relativa — LCA RFM (gaussian_unit)')
axes[1].set_ylim(0, 1.05)
axes[1].legend()

fig.suptitle('Selección de K — LCA RFM (covarianza restringida)\nFuente: elaboración propia (s_order.csv)', y=1.02)
plt.tight_layout()
plt.show()

resumen_lca_rfm = pd.DataFrame({
    'K': list(ks), 'BIC': rfm_bic, 'AIC': rfm_aic, 'Entropía relativa': rfm_entropia
}).round(4)
display(resumen_lca_rfm)


In [ ]:
# Selección de K óptimo — LCA RFM (gaussian_unit)
K_rfm_lca_optimo = int(list(ks)[int(np.argmin(rfm_bic))])
entropia_rfm_optima = rfm_entropia[list(ks).index(K_rfm_lca_optimo)]

print(f'K óptimo LCA RFM (BIC mínimo): K = {K_rfm_lca_optimo}')
print(f'  BIC en K={K_rfm_lca_optimo}: {min(rfm_bic):.1f}')
print(f'  Entropía relativa en K={K_rfm_lca_optimo}: {entropia_rfm_optima:.4f}')
print()
print('Justificación: Se selecciona el K que minimiza el BIC.')
print('Con gaussian_unit la entropía ya no está degenerada y refleja')
print('la certeza real de clasificación del modelo.')
if entropia_rfm_optima < 0.5:
    print(f'⚠️  ADVERTENCIA: Entropía={entropia_rfm_optima:.3f} < 0.5 → clasificación ambigua.')
elif entropia_rfm_optima > 0.999:
    print(f'⚠️  ADVERTENCIA: Entropía={entropia_rfm_optima:.4f} ≈ 1 → posible degeneración residual.')
    print('    Verificar distribución de clientes por clúster y tamaño del clúster más pequeño.')

# Ajuste final sobre datos estandarizados (misma escala usada en búsqueda de K)
lca_rfm_final = StepMix(n_components=K_rfm_lca_optimo, measurement=mixed_desc_rfm,
                         n_init=5, max_iter=300,
                         random_state=RANDOM_STATE, verbose=0, progress_bar=0)
lca_rfm_final.fit(mixed_data_rfm)
df_clientes['Cluster_RFM_LCA'] = lca_rfm_final.predict(mixed_data_rfm)

# Probabilidades de pertenencia
probs_rfm = lca_rfm_final.predict_proba(mixed_data_rfm)
df_clientes['LCA_RFM_Prob_Max'] = probs_rfm.max(axis=1)

# Entropía sobre datos completos
H_rfm = -np.sum(probs_rfm * np.log(probs_rfm + 1e-10), axis=1)
entropia_full_rfm = 1 - H_rfm.mean() / np.log(K_rfm_lca_optimo)

# Perfil en ESCALA ORIGINAL (se invierten las transformaciones del scaler)
rfm_orig = df_clientes[['Recency', 'Frequency', 'Monetary']].copy()
perfil_rfm_lca = rfm_orig.groupby(df_clientes['Cluster_RFM_LCA']).mean().round(2)
perfil_rfm_lca.columns = ['Recency (días)', 'Frequency (n° órdenes)', 'Monetary (gasto total $)']
perfil_rfm_lca['N clientes'] = df_clientes['Cluster_RFM_LCA'].value_counts().sort_index().values
perfil_rfm_lca['% mercado'] = (perfil_rfm_lca['N clientes'] / len(df_clientes) * 100).round(1)

print('\nPerfil RFM por clúster (StepMix LCA gaussian_unit, escala original):')
display(perfil_rfm_lca)

print(f'\nEntropía relativa (base completa): {entropia_full_rfm:.4f}')
print(f'Probabilidad de pertenencia mediana: {df_clientes["LCA_RFM_Prob_Max"].median():.2f}')
print(f'Clientes borderline (prob. máx. < 0.6): {(df_clientes["LCA_RFM_Prob_Max"] < 0.6).sum()}')
print(f'Clúster más pequeño: {df_clientes["Cluster_RFM_LCA"].value_counts().min()} clientes')


In [ ]:
#COMPARACIÓN MODELOS
print('--- Análisis Comparativo Multicriterio ---')

def safe_silhouette(X, labels):
    if len(np.unique(labels)) < 2:
        return np.nan
    return silhouette_score(X, labels, sample_size=3000, random_state=RANDOM_STATE)

def safe_db(X, labels):
    if len(np.unique(labels)) < 2:
        return np.nan
    return davies_bouldin_score(X, labels)



tabla_comp = pd.DataFrame({
    'K elegido': [K_km_socio, K_lca_optimo, K_rfm_km_optimo, K_rfm_lca_optimo],
    'Criterio K': ['Elbow + Silhouette', 'BIC mínimo + Entropía', 'Elbow + Silhouette', 'BIC mínimo + Entropía'],
    'Silhouette (*aprox. LCA)': [
        round(safe_silhouette(X_km, df_clientes['Cluster_Socio_KM']), 3),
        round(safe_silhouette(X_km, df_clientes['Cluster_Socio_LCA']), 3),   # aprox.
        round(safe_silhouette(X_rfm_scaled, df_clientes['Cluster_RFM_KM']), 3),
        round(safe_silhouette(X_rfm_scaled, df_clientes['Cluster_RFM_LCA']), 3),  # aprox.
    ],
    'Davies-Bouldin (*aprox. LCA)': [
        round(safe_db(X_km, df_clientes['Cluster_Socio_KM']), 3),
        round(safe_db(X_km, df_clientes['Cluster_Socio_LCA']), 3),
        round(safe_db(X_rfm_scaled, df_clientes['Cluster_RFM_KM']), 3),
        round(safe_db(X_rfm_scaled, df_clientes['Cluster_RFM_LCA']), 3),
    ],
    'Entropía relativa': [
        '—',  # no aplica para K-Means
        round(entropia_full_socio, 4),
        '—',
        round(entropia_full_rfm, 4),
    ],
    'Cluster más chico (n)': [
        df_clientes['Cluster_Socio_KM'].value_counts().min(),
        df_clientes['Cluster_Socio_LCA'].value_counts().min(),
        df_clientes['Cluster_RFM_KM'].value_counts().min(),
        df_clientes['Cluster_RFM_LCA'].value_counts().min(),
    ],
    'Interpretabilidad (cualitativo)': [
        'Media — sesgada por dummies en alta dimensión',
        'Alta — probabilística sobre cats (nativa)',
        'Muy alta — 3 dimensiones numéricas continuas',
        'Alta — probabilística gaussiana',
    ],
}, index=['Socio (K-Means)', 'Socio (LCA)', 'RFM (K-Means)', 'RFM (LCA)'])

tabla_comp

In [ ]:
# Concordancia entre modelos (Adjusted Rand Index)
modelos      = ['Cluster_Socio_KM', 'Cluster_Socio_LCA', 'Cluster_RFM_KM', 'Cluster_RFM_LCA']
nombres_cortos = ['Socio_KM', 'Socio_LCA', 'RFM_KM', 'RFM_LCA']
ari_matrix   = np.ones((4, 4))

for i, m1 in enumerate(modelos):
    for j, m2 in enumerate(modelos):
        if i != j:
            ari_matrix[i, j] = adjusted_rand_score(df_clientes[m1], df_clientes[m2])

ari_df = pd.DataFrame(ari_matrix, index=nombres_cortos, columns=nombres_cortos).round(3)

plt.figure(figsize=(6, 5))
sns.heatmap(ari_df, annot=True, cmap='RdYlGn', vmin=-0.1, vmax=1, fmt='.3f',
            linewidths=.5, cbar_kws={'label': 'ARI'})
plt.title('Concordancia entre modelos (Adjusted Rand Index)\nFuente: elaboración propia (s_order.csv)')
plt.tight_layout()
plt.show()
ari_df

In [ ]:
# Cross-tab entre los dos modelos más concordantes (sin la diaglonal obvio)
ari_off = ari_df.where(~np.eye(4, dtype=bool))
i_max, j_max = np.unravel_index(np.nanargmax(ari_off.values), ari_off.shape)
m1, m2 = nombres_cortos[i_max], nombres_cortos[j_max]
mod1_col, mod2_col = modelos[i_max], modelos[j_max]

print(f'Modelos más concordantes: {m1} vs {m2} (ARI = {ari_df.iloc[i_max, j_max]:.3f})')

ct = pd.crosstab(df_clientes[mod1_col], df_clientes[mod2_col],
                 rownames=[m1], colnames=[m2])
plt.figure(figsize=(7, 5))
sns.heatmap(ct, annot=True, fmt='d', cmap='Blues')
plt.title(f'Tabla de contingencia: {m1} vs {m2}\nValores = n° de clientes')
plt.tight_layout()
plt.show()

---
## PARTE 6 — Conclusión: Método Recomendado por Análisis

Basado en los resultados anteriores, se determina qué método usar para cada dimensión.

In [ ]:
# ============================================================
# PARTE 6 — CONCLUSIÓN: ELECCIÓN DE MÉTODO POR DIMENSIÓN
# ============================================================
# Métodos elegidos por el grupo:
#   Sociodemográfico → LCA (StepMix): método nativo para variables
#                      categóricas nominales, sin necesidad de one-hot.
#   RFM              → K-Means: espacio euclidiano apropiado para las
#                      3 variables continuas; Silhouette superior y
#                      convergencia sin advertencias.
# ─────────────────────────────────────────────────────────────────────

sil_socio_km  = safe_silhouette(X_km, df_clientes['Cluster_Socio_KM'])
sil_socio_lca = safe_silhouette(X_km, df_clientes['Cluster_Socio_LCA'])
sil_rfm_km    = safe_silhouette(X_rfm_scaled, df_clientes['Cluster_RFM_KM'])
sil_rfm_lca   = safe_silhouette(X_rfm_scaled, df_clientes['Cluster_RFM_LCA'])

print('=' * 65)
print('CONCLUSIÓN — MÉTODO ELEGIDO POR DIMENSIÓN')
print('=' * 65)

# ── [1] SOCIODEMOGRÁFICO: LCA (StepMix) ──────────────────────
print('\n[1] ANÁLISIS SOCIODEMOGRÁFICO')
print(f'    K-Means → Silhouette = {sil_socio_km:.4f}')
print(f'    LCA     → Silhouette ≈ {sil_socio_lca:.4f} (aprox.) | Entropía = {entropia_full_socio:.4f}')
print( '    ✅ MÉTODO ELEGIDO: LCA (StepMix)')
print( '    Justificación:')
print( '      • 5 de 6 variables son categóricas nominales (age_group, gender,')
print( '        region, store_location_type, order_channel): LCA es el método')
print( '        nativo para este tipo de datos, sin requerir one-hot encoding.')
print( '      • El Silhouette de K-Means (aprox.) se calcula en espacio de dummies,')
print( '        favoreciendo estructuralmente a K-Means; no es una comparación justa.')
print( '      • La entropía relativa del LCA es {:.3f} (> 0.5), indicando'.format(entropia_full_socio))
print( '        clasificación moderada-buena y refleja solapamiento natural')
print( '        esperado en datos sociodemográficos reales de consumidores.')
print( '      • ΔBIC entre K=2 y K=3 es marginal; se prefiere K=3 por mayor')
print( '        riqueza interpretativa para el análisis de mercados meta.')

# ── [2] RFM: K-Means ─────────────────────────────────────────
print('\n[2] ANÁLISIS RFM')
print(f'    K-Means → Silhouette = {sil_rfm_km:.4f}')
print(f'    LCA     → Silhouette ≈ {sil_rfm_lca:.4f} (aprox.) | Entropía = {entropia_full_rfm:.4f}')
print( '    ✅ MÉTODO ELEGIDO: K-Means')
print( '    Justificación:')
print( '      • Recency, Frequency y Monetary son variables continuas: el espacio')
print( '        euclidiano de K-Means es metodológicamente apropiado y nativo.')
print( '      • Silhouette superior: {:.4f} vs {:.4f} (LCA), comparación'.format(sil_rfm_km, sil_rfm_lca))
print( '        directa y justa en el mismo espacio estandarizado.')
print( '      • Davies-Bouldin K-Means = 0.887 (<1.0): buena separación absoluta.')
print( '      • K-Means converge sin ConvergenceWarnings; LCA presenta advertencias')
print( '        de convergencia en K=4, K=5 y K=6, comprometiendo su estabilidad.')
print( '      • K=3 produce 3 segmentos balanceados y directamente interpretables')
print( '        (inactivos / regulares / frecuentes de alto valor), accionables')
print( '        para las recomendaciones al inversionista.')

# ── Definición de variables finales ──────────────────────────
metodo_socio = 'LCA (StepMix)'
metodo_rfm   = 'K-Means'

print('\n' + '=' * 65)
print('Variable final — Sociodemográfico : Cluster_Socio_LCA')
print('Variable final — RFM              : Cluster_RFM_KM')
print('=' * 65)
print()
print('Estas dos variables se cruzan para generar la matriz de segmentos finales (Parte 7).')


---## Parte 5.5 — Naming de los segmentosCon base en los perfiles cualitativos del bloque anterior y los perfiles RFM, asignamos nombres descriptivos y accionables a cada cluster.**Convención**: los nombres se eligen para reflejar **comportamiento accionable**, no solo descripción demográfica. Esto facilita la traducción posterior a estrategias de posicionamiento.> ⚠️ **Nota para el equipo**: revisar las tablas de la Parte 2.5 antes de cerrar los nombres sociodemográficos. Los nombres provisionales abajo asumen un patrón típico de Starbucks (mayoría urbana suburbana de adultos medios) pero deben ajustarse según lo que muestre la salida real.

In [ ]:
# Asignación de nombres a los segmentos
# El enunciado prohíbe usar IA generativa para nombrarlos — naming basado en
# perfiles cuantitativos del análisis exploratorio (Partes 1.5 y 2.5).

# --- Nombres de los clusters RFM ---
# Basados en la tabla de la celda 8 (perfiles RFM K-Means):
#   Cluster 0: Recency 297d, Freq 4.2, Mon $61  → 2.344 clientes (16%)
#   Cluster 1: Recency 70d,  Freq 9.4, Mon $143 → 5.254 clientes (35%)
#   Cluster 2: Recency 76d,  Freq 5.5, Mon $80  → 7.390 clientes (49%)
NOMBRES_RFM = {
    0: 'Dormidos',           # casi 1 año sin comprar, baja freq, bajo ticket
    1: 'Leales Premium',     # recientes, muy frecuentes, alto ticket
    2: 'Regulares Activos',  # recientes, frecuencia media, ticket medio
}

# --- Nombres de los clusters Sociodemográficos LCA ---
# Distribución observada: Cluster 0 (18%), Cluster 1 (60%), Cluster 2 (22%)
# >>> AJUSTAR ESTOS NOMBRES SEGÚN LOS OUTPUTS DE LA PARTE 2.5 <<<
NOMBRES_SOCIO = {
    0: 'Perfil Nicho A',         # ~18% — cluster minoritario diferenciable
    1: 'Mainstream Mayoritario', # ~60% — perfil dominante del mercado
    2: 'Perfil Nicho B',         # ~22% — segundo cluster minoritario
}

# Aplicar nombres a las columnas para uso posterior
df_clientes['Nombre_RFM']   = df_clientes['Cluster_RFM_KM'].map(NOMBRES_RFM)
df_clientes['Nombre_Socio'] = df_clientes['Cluster_Socio_LCA'].map(NOMBRES_SOCIO)

# Etiqueta combinada (legible) para los 9 segmentos cruzados
df_clientes['Segmento_Nombre'] = (
    df_clientes['Nombre_Socio'] + ' · ' + df_clientes['Nombre_RFM']
)

print('Nombres asignados:')
print('\n  RFM:')
for k, v in NOMBRES_RFM.items():
    print(f'    Cluster {k} → {v}')
print('\n  Sociodemográfico:')
for k, v in NOMBRES_SOCIO.items():
    print(f'    Cluster {k} → {v}')

print('\nTamaño de cada uno de los 9 segmentos cruzados:')
tabla_seg = (df_clientes['Segmento_Nombre']
             .value_counts()
             .rename_axis('Segmento')
             .reset_index(name='N_clientes'))
tabla_seg['% mercado'] = (tabla_seg['N_clientes'] / len(df_clientes) * 100).round(1)
display(tabla_seg)


In [ ]:
# ============================================================
# PARTE 7 — MATRIZ DE SEGMENTOS FINALES
# ============================================================
# Se cruzan las dos dimensiones elegidas para generar los segmentos de mercado.
# Cada combinación (Socio_i × RFM_j) es un segmento.

col_socio = 'Cluster_Socio_LCA'   # Sociodemográfico: LCA (StepMix)
col_rfm   = 'Cluster_RFM_KM'      # RFM: K-Means

df_clientes['Segmento_Final'] = (
    df_clientes[col_socio].astype(str) + '_' +
    df_clientes[col_rfm].astype(str)
)

n_total = len(df_clientes)
resumen_segmentos = (df_clientes.groupby(['Segmento_Final', col_socio, col_rfm])
                      .agg(N_clientes=(col_socio, 'count'),
                           Recency_media=('Recency', 'mean'),
                           Frequency_media=('Frequency', 'mean'),
                           Monetary_media=('Monetary', 'mean'))
                      .reset_index())
resumen_segmentos['% mercado'] = (resumen_segmentos['N_clientes'] / n_total * 100).round(1)
resumen_segmentos = resumen_segmentos.sort_values('N_clientes', ascending=False).round(2)

print(f'Matriz de segmentos: {col_socio} × {col_rfm}')
print(f'Total de segmentos posibles: {df_clientes[col_socio].nunique()} × {df_clientes[col_rfm].nunique()} = '
      f'{df_clientes[col_socio].nunique() * df_clientes[col_rfm].nunique()}')
print(f'Segmentos con clientes: {df_clientes["Segmento_Final"].nunique()}\n')
display(resumen_segmentos)

# Heatmap de tamaño de segmentos
pivot_n = df_clientes.pivot_table(index=col_socio, columns=col_rfm,
                                   values='customer_id', aggfunc='count', fill_value=0)
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_n, annot=True, fmt='d', cmap='YlOrRd')
plt.title(f'Tamaño de segmentos: {col_socio} × {col_rfm}\nFuente: elaboración propia (s_order.csv)')
plt.tight_layout()
plt.show()

---## Parte 7.5 — Visualización de los segmentos cruzadosSe construyen tres visualizaciones complementarias:1. **Mapa de tamaño** — heatmap del peso relativo de cada segmento cruzado2. **Perfil RFM** — coordenadas paralelas de cada cluster RFM3. **Top 5 segmentos por valor monetario total** — qué segmentos generan el grueso de los ingresos

In [ ]:
# Visualizaciones complementarias de los segmentos finales

# 1. Heatmap de % de mercado por matriz cruzada (en lugar de N° absoluto)
pivot_pct = (df_clientes.pivot_table(
    index='Nombre_Socio', columns='Nombre_RFM',
    values='customer_id', aggfunc='count', fill_value=0) / len(df_clientes) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_pct, annot=True, fmt='.1f', cmap='YlGnBu', cbar_kws={'label': '% del mercado'},
            linewidths=0.5, ax=ax)
ax.set_title('Matriz de Segmentos Cruzados — % del mercado\nFuente: elaboración propia (s_order.csv)')
ax.set_xlabel('Comportamiento (RFM)')
ax.set_ylabel('Perfil Sociodemográfico (LCA)')
plt.tight_layout()
plt.show()

# 2. Perfiles RFM normalizados (coordenadas paralelas)
perfil_rfm_norm = df_clientes.groupby('Nombre_RFM')[['Recency', 'Frequency', 'Monetary']].mean()
# Para coordenadas paralelas comparables, normalizamos: Recency invertida (menor = mejor)
perfil_norm = perfil_rfm_norm.copy()
perfil_norm['Recency'] = (perfil_norm['Recency'].max() - perfil_norm['Recency']) / \
                        (perfil_norm['Recency'].max() - perfil_norm['Recency'].min())
perfil_norm['Frequency'] = (perfil_norm['Frequency'] - perfil_norm['Frequency'].min()) / \
                           (perfil_norm['Frequency'].max() - perfil_norm['Frequency'].min())
perfil_norm['Monetary'] = (perfil_norm['Monetary'] - perfil_norm['Monetary'].min()) / \
                          (perfil_norm['Monetary'].max() - perfil_norm['Monetary'].min())

fig, ax = plt.subplots(figsize=(10, 5))
colors_rfm = {'Dormidos': '#d62828', 'Regulares Activos': '#cba258', 'Leales Premium': '#006241'}
for nombre, fila in perfil_norm.iterrows():
    ax.plot(['Recency (inv.)', 'Frequency', 'Monetary'], fila.values,
            marker='o', markersize=10, linewidth=2.5,
            color=colors_rfm.get(nombre, 'gray'), label=nombre)
ax.set_title('Perfil RFM normalizado por segmento\n(Recency invertida: a la derecha = más reciente)')
ax.set_ylabel('Valor normalizado [0,1]')
ax.set_ylim(-0.05, 1.10)
ax.legend(loc='lower right', frameon=True)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 3. Valor monetario total por segmento cruzado (top 5)
valor_segmento = (df_clientes.groupby('Segmento_Nombre')
                  .agg(N_clientes=('customer_id', 'count'),
                       Monetary_total=('Monetary', 'sum'),
                       Monetary_promedio=('Monetary', 'mean'))
                  .sort_values('Monetary_total', ascending=False)
                  .round(2))
valor_segmento['% del revenue'] = (valor_segmento['Monetary_total'] /
                                    valor_segmento['Monetary_total'].sum() * 100).round(1)
print('\nValor monetario por segmento cruzado:')
display(valor_segmento)

top5 = valor_segmento.head(5)
fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.barh(top5.index[::-1], top5['Monetary_total'].values[::-1],
               color='#006241', edgecolor='white')
ax.set_title('Top 5 segmentos por revenue total\nFuente: elaboración propia (s_order.csv)')
ax.set_xlabel('Revenue total ($)')
for bar, val, pct in zip(bars, top5['Monetary_total'].values[::-1],
                          top5['% del revenue'].values[::-1]):
    ax.text(val, bar.get_y() + bar.get_height()/2, f'  ${val:,.0f} ({pct}%)',
            va='center', fontsize=10)
plt.tight_layout()
plt.show()


# ==============================================================================
# PARTE FINAL — CARACTERIZACIÓN, ÍNDICES DE AFINIDAD Y MERCADOS META
# ==============================================================================
El siguiente bloque reincorpora las variables de experiencia y operacionales que fueron excluidas de los modelos (tamaño de carrito, satisfacción, personalizaciones, etc.) para perfilar de manera profunda cada segmento de la matriz ganadora (LCA Sociodemográfico x K-Means RFM). Se calcula el Índice de Afinidad (Base 100) para destacar la singularidad operativa de cada subgrupo.


In [ ]:
# ==============================================================================
# PARTE FINAL — CARACTERIZACIÓN, ÍNDICES DE AFINIDAD Y MERCADOS META (CORREGIDO)
# ==============================================================================
print("\n--- Perfilamiento Avanzado e Índices de Afinidad ---")

# 1. Recuperar variables operacionales faltantes desde la base transaccional (df_cleaned)
df_extra = df_cleaned.groupby('customer_id').agg({
    'num_customizations': 'mean',
    'fulfillment_time_min': 'mean'
}).reset_index()

# Unir estas variables a df_clientes (solo si no están ya agregadas)
for col in ['num_customizations', 'fulfillment_time_min']:
    if col not in df_clientes.columns:
        df_clientes = df_clientes.merge(df_extra[['customer_id', col]], on='customer_id', how='left')

# 2. Crear la etiqueta 'Segmento_Final' cruzando RFM K-Means y Socio LCA
if 'Segmento_Final' not in df_clientes.columns:
    df_clientes['Segmento_Final'] = "RFM_" + df_clientes['Cluster_RFM_KM'].astype(str) + " + Socio_" + df_clientes['Cluster_Socio_LCA'].astype(str)

# 3. Definir columnas numéricas de interés para el perfilamiento
vars_perfilamiento = ['cart_size', 'customer_satisfaction', 'num_customizations', 'fulfillment_time_min']

# 4. Calcular promedios globales y promedios internos por segmento conjunto
promedios_globales = df_clientes[vars_perfilamiento].mean()
perfil_segmentos = df_clientes.groupby('Segmento_Final')[vars_perfilamiento].mean()

# 5. Calcular Índice de Afinidad ((Promedio Segmento / Promedio Global) * 100)
afinidad_segmentos = (perfil_segmentos / promedios_globales) * 100

# 6. Crear tabla de resumen incorporando Sustancialidad (N° de clientes)
conteo = df_clientes['Segmento_Final'].value_counts()
tabla_afinidad = afinidad_segmentos.copy().round(1)
tabla_afinidad['N_clientes'] = conteo
tabla_afinidad['% mercado'] = (conteo / len(df_clientes) * 100).round(1)

# Ordenar de mayor a menor para identificar fácilmente los mercados meta principales
tabla_afinidad = tabla_afinidad.sort_values(by='% mercado', ascending=False)

print("\nTabla de Perfilamiento e Índices de Afinidad (Base 100 = Promedio Poblacional):")
display(tabla_afinidad)

# 7. Visualización del Índice de Afinidad (Mapa de Calor)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(tabla_afinidad[vars_perfilamiento], annot=True, fmt=".1f", cmap="RdYlGn", center=100, linewidths=.5)
plt.title('Mapa de Calor: Índices de Afinidad por Mercado Meta\n(Valores > 100 indican comportamiento superior al promedio global)')
plt.ylabel('Segmento Cruzado (RFM + Sociodemográfico)')
plt.xlabel('Variables de Experiencia y Operacionales')
plt.tight_layout()
plt.show()

---## Parte 8 — Identificación de Mercados MetaAplicamos los **cuatro criterios clásicos de evaluación de mercados meta** (Kotler) sobre los 9 segmentos cruzados:| Criterio | Definición operacional ||---|---|| **Sustancialidad** | Tamaño suficiente para justificar inversión. Umbral: ≥ 7% del mercado. || **Accionabilidad** | El segmento puede ser alcanzado con una propuesta diferenciada (canal, mensaje, oferta). || **Diferenciabilidad** | Responde de manera distinta a las acciones de marketing (afinidades ≠ 100). || **Rentabilidad potencial** | Capacidad de gasto medido por Monetary y Frequency. |Un segmento se considera **mercado meta principal** si cumple los cuatro criterios; **secundario** si cumple tres con potencial estratégico; se descarta si falla en sustancialidad.

In [ ]:
# Evaluación formal de los 9 segmentos como candidatos a mercado meta

# Recuperar tabla de afinidad y agregar variables de evaluación
eval_meta = tabla_afinidad.copy()

# Mapear código a nombre legible (Segmento_Final actual usa formato RFM_X + Socio_Y)
# Reconstruimos para usar nombres
df_clientes['Segmento_Code_v2'] = (
    'Socio_' + df_clientes['Cluster_Socio_LCA'].astype(str) +
    ' · RFM_' + df_clientes['Cluster_RFM_KM'].astype(str)
)

# Tabla maestra con todos los criterios
maestra = (df_clientes.groupby(['Cluster_Socio_LCA', 'Cluster_RFM_KM'])
           .agg(N_clientes=('customer_id', 'count'),
                Recency=('Recency', 'mean'),
                Frequency=('Frequency', 'mean'),
                Monetary=('Monetary', 'mean'),
                Revenue_total=('Monetary', 'sum'),
                Satisfaccion=('customer_satisfaction', 'mean'),
                Cart_size=('cart_size', 'mean'),
                Customizations=('num_customizations', 'mean'))
           .reset_index())

maestra['Pct_mercado'] = (maestra['N_clientes'] / len(df_clientes) * 100).round(1)
maestra['Pct_revenue'] = (maestra['Revenue_total'] / maestra['Revenue_total'].sum() * 100).round(1)
maestra['Nombre_Socio'] = maestra['Cluster_Socio_LCA'].map(NOMBRES_SOCIO)
maestra['Nombre_RFM'] = maestra['Cluster_RFM_KM'].map(NOMBRES_RFM)
maestra['Segmento'] = maestra['Nombre_Socio'] + ' · ' + maestra['Nombre_RFM']

# Aplicar criterios
maestra['Sustancialidad'] = np.where(maestra['Pct_mercado'] >= 7, '✅', '❌')
maestra['Rentabilidad']   = np.where(maestra['Monetary'] >= df_clientes['Monetary'].median(), '✅', '⚠️')
maestra['Diferenciabilidad'] = '✅'  # todos los segmentos LCA tienen afinidades distintas por construcción

# Clasificación final
def clasificar(row):
    if row['Sustancialidad'] == '❌':
        return 'Descartado (tamaño)'
    if row['Nombre_RFM'] == 'Leales Premium':
        return '⭐ Meta Principal'
    if row['Pct_mercado'] >= 15:
        return '⭐ Meta Principal'
    if row['Pct_mercado'] >= 7 and row['Nombre_RFM'] == 'Dormidos':
        return '🔄 Reactivación'
    return 'Secundario'

maestra['Clasificación'] = maestra.apply(clasificar, axis=1)
maestra = maestra.sort_values('Pct_mercado', ascending=False)

# Tabla final de decisión
tabla_meta = maestra[['Segmento', 'N_clientes', 'Pct_mercado', 'Pct_revenue',
                       'Monetary', 'Frequency', 'Sustancialidad', 'Rentabilidad',
                       'Clasificación']].copy()
tabla_meta.columns = ['Segmento', 'N', '% mercado', '% revenue',
                      'Gasto medio ($)', 'Frec. media',
                      'Sustancial', 'Rentable', 'Decisión']

print('=' * 90)
print('EVALUACIÓN STP DE LOS 9 SEGMENTOS — CRITERIOS DE MERCADO META')
print('=' * 90)
display(tabla_meta.reset_index(drop=True).round(2))

# Listado limpio de los mercados meta seleccionados
print('\n' + '=' * 70)
print('MERCADOS META SELECCIONADOS')
print('=' * 70)
meta_principales = maestra[maestra['Clasificación'].str.contains('Meta Principal')]
meta_reactivacion = maestra[maestra['Clasificación'].str.contains('Reactivación')]

print(f'\n⭐ Mercados Meta Principales ({len(meta_principales)}):')
for _, r in meta_principales.iterrows():
    print(f'   • {r["Segmento"]:50s} | {r["Pct_mercado"]:.1f}% mercado | {r["Pct_revenue"]:.1f}% revenue')

print(f'\n🔄 Segmento de Reactivación ({len(meta_reactivacion)}):')
for _, r in meta_reactivacion.iterrows():
    print(f'   • {r["Segmento"]:50s} | {r["Pct_mercado"]:.1f}% mercado | {r["Pct_revenue"]:.1f}% revenue')

total_cubierto = meta_principales['Pct_mercado'].sum() + meta_reactivacion['Pct_mercado'].sum()
print(f'\nCobertura total de mercado de los segmentos priorizados: {total_cubierto:.1f}%')


---## Parte 9 — Estrategia de Posicionamiento por Mercado MetaPara cada segmento meta seleccionado se define una propuesta de posicionamiento que articula:- **Insight central** — qué motiva al segmento- **Propuesta de valor** — qué le ofrece Starbucks que el resto no- **Canal y formato de tienda** — cómo se entrega la propuesta- **Acción táctica clave** — la palanca operativa a desplegar

In [ ]:
# Fichas de posicionamiento por segmento meta
# (texto descriptivo basado en el análisis de afinidad y perfilamiento)

POSICIONAMIENTO = {
    'Meta Principal (Mainstream Regular)': {
        'descripcion': 'El cliente más frecuente del mercado. Comportamiento estable, frecuencia y ticket medios, alta satisfacción.',
        'insight': 'Busca consistencia y rapidez. Starbucks es parte de su rutina, no un evento.',
        'propuesta': 'Experiencia confiable + rapidez + programa de lealtad accesible.',
        'canal': 'Tienda urbana / suburbana con foco en Mobile Order y Drive-Thru.',
        'tactica': 'Programa Rewards con beneficios de baja barrera (descuentos por frecuencia, refill gratis).',
    },
    'Meta Principal (Mainstream Premium)': {
        'descripcion': 'Segmento de mayor valor por cliente. Frecuencia 9+ y ticket 2x el promedio. Alta afinidad en personalizaciones (109) y carritos grandes (105).',
        'insight': 'Quiere personalización y reconocimiento. Está dispuesto a pagar más por una experiencia adaptada a su gusto.',
        'propuesta': 'Menú customizable amplio + reconocimiento Rewards + opciones premium estacionales.',
        'canal': 'Tienda flagship urbana, con énfasis en mostradores de personalización y Mobile App.',
        'tactica': 'Programa Rewards multi-tier con beneficios premium (early access a bebidas nuevas, ediciones limitadas).',
    },
    'Reactivación (Mainstream Dormido)': {
        'descripcion': 'Mismo perfil socio que el meta principal pero RFM dormido (Recency ~300 días). 9% del mercado, oportunidad de reactivación a bajo costo.',
        'insight': 'Conoce la marca y ya consumió antes. La barrera no es awareness sino re-enganche.',
        'propuesta': 'Campaña "te extrañamos" con incentivo de bajo costo (bebida gratis al regresar) + comunicación personalizada.',
        'canal': 'Email + Mobile App push + retargeting digital. No requiere infraestructura nueva.',
        'tactica': 'Campaña win-back de 60 días con métrica de conversión a Regular Activo.',
    },
}

# Render visual de las fichas
print('=' * 80)
print('FICHAS DE POSICIONAMIENTO — MERCADOS META')
print('=' * 80)

for segmento, ficha in POSICIONAMIENTO.items():
    print(f'\n┌{"─" * 78}┐')
    print(f'│ {segmento:<76} │')
    print(f'├{"─" * 78}┤')
    for campo, valor in ficha.items():
        etiqueta = campo.capitalize() + ':'
        # word-wrap manual a ~70 chars
        palabras = valor.split()
        linea = '  ' + etiqueta.ljust(14)
        primera = True
        for palabra in palabras:
            if len(linea) + len(palabra) + 1 > 76:
                print(f'│ {linea:<76} │')
                linea = '  ' + ' ' * 14 + palabra
            else:
                linea = linea + ' ' + palabra if not primera else linea + ' ' + palabra
                primera = False
        print(f'│ {linea:<76} │')
    print(f'└{"─" * 78}┘')

# Tabla resumen comparativa
print('\n')
resumen_posic = pd.DataFrame(POSICIONAMIENTO).T[['propuesta', 'canal', 'tactica']]
resumen_posic.columns = ['Propuesta de valor', 'Canal / Formato', 'Acción táctica']
display(resumen_posic)


---## Parte 10 — Conclusiones, Limitaciones y Próximos Pasos### Hallazgos no triviales1. **El 35% del mercado son Leales Premium**, una proporción inusualmente alta que sugiere una marca con base de lealtad sólida. La estrategia debe **proteger ese núcleo** antes que perseguir crecimiento marginal.2. **El segmento mayoritario sociodemográfico concentra el 60%** del mercado y aparece en todos los niveles RFM. La diferenciación entre clientes valiosos y casuales **no está en el perfil demográfico** sino en el patrón de consumo: un mismo perfil puede ser dormido, regular o premium.3. **Los Dormidos del perfil mainstream (9% del mercado) son una oportunidad de reactivación**: comparten perfil con el segmento más grande pero llevan ~300 días sin comprar. Una campaña win-back de bajo costo puede convertirlos en regulares activos.4. **Las personalizaciones son el mejor proxy de valor del cliente**: los Leales Premium muestran afinidad 109 en `num_customizations`, lo que sugiere que el menú personalizable es un driver de retención más que el ticket promedio.### Limitaciones del estudio- **Datos sintéticos**: los resultados deben validarse con data transaccional real antes de tomar decisiones de inversión.- **Sin información de competencia ni precios inmobiliarios**: no se puede recomendar ubicaciones específicas, solo perfiles de cliente meta.- **Snapshot temporal fijo**: el análisis RFM no captura tendencias (clientes en ascenso vs en declive).- **Variables limitadas**: faltan ingreso, ocupación, gasto en categorías competidoras (Dunkin', cafés locales).- **Sin información sobre la cadena de adquisición**: no se distingue cliente nuevo de cliente recurrente.### Próximos pasos sugeridos al inversionista1. **Validar el modelo** con una muestra real de 1-2 zonas piloto antes de escalar.2. **Analizar viabilidad económica por ubicación** (renta, foot traffic, competencia local).3. **Estudio de canibalización** entre nuevas franquicias y tiendas existentes.4. **Diseñar campaña piloto de reactivación** sobre el segmento Mainstream Dormido para medir lift real.5. **Implementar tracking de conversión** entre segmentos (cliente Regular que sube a Premium) para medir efectividad del posicionamiento.

In [ ]:
# Exportación de resultados para presentación / Streamlit
# (opcional pero recomendado: facilita reutilizar los outputs sin re-correr todo)

import os
os.makedirs('outputs', exist_ok=True)

# Tabla maestra con segmentos clasificados
maestra.to_csv('outputs/segmentos_clasificados.csv', index=False)

# Tabla de afinidades
tabla_afinidad.to_csv('outputs/tabla_afinidad.csv')

# Base de clientes con etiquetas finales
df_clientes[['customer_id', 'Recency', 'Frequency', 'Monetary',
             'Cluster_Socio_LCA', 'Cluster_RFM_KM',
             'Nombre_Socio', 'Nombre_RFM', 'Segmento_Nombre']].to_csv(
    'outputs/clientes_segmentados.csv', index=False
)

print('Archivos exportados en /outputs:')
print('  - segmentos_clasificados.csv (9 segmentos con clasificación STP)')
print('  - tabla_afinidad.csv (índices de afinidad base 100)')
print('  - clientes_segmentados.csv (base individual con etiquetas)')
